# The Regression Line
The correlation coefficient $r$ doesn't just measure how clustered the points in a scatter plot are about a straight line. It also helps identify the straight line about which the points are clustered. In this section we will retrace the path that Galton and Pearson took to discover that line.

As we have seen, our dataset on the heights of parents and their adult children indicates a linear association between the two variables. The linearity was confirmed when our predictions of the children's heights based on the midparent heights roughly followed a straight line.

In [1]:
import polars as pl
import numpy as np
import hvplot.polars
import holoviews as hv
path_data = '../../../assets/data/'


In [2]:
original = pl.read_csv(path_data + 'family_heights.csv', infer_schema_length=10000)

heights = pl.DataFrame([
    original.get_column('midparentHeight').alias('MidParent'),
    original.get_column('childHeight').alias('Child')
    ])

In [3]:
def predict_child(mpht):
    """Return a prediction of the height of a child 
    whose parents have a midparent height of mpht.
    
    The prediction is the average height of the children 
    whose midparent height is in the range mpht plus or minus 0.5 inches.
    """
    
    close_points = heights.filter(pl.col('MidParent').is_between(mpht-0.5, mpht + 0.5, closed="left"))
    return close_points.get_column('Child').mean()   

In [4]:
heights_with_predictions = heights.with_columns(
    heights.get_column('MidParent').map_elements(predict_child, return_dtype=float).alias('Prediction')
    )

In [5]:
heights_with_predictions.hvplot.scatter(x='MidParent', y=['Child', 'Prediction'], color=['#002fa7', '#706700'])

Scatterplot with 'MidParent' on the x-axis. The previous data points are shown again in dark blue and labeled 'Child.' These dark blue data points tend to have a positive correlation; as the x value increases so does the y value. There are additionally gold data points labeled 'Prediction.' The gold data points are in the center of the blue data points and also show a positive association. The x-axis ranges from 64 to 74. The y-axis ranges from 55 to 80.

## Measuring in Standard Units

Let's see if we can find a way to identify this line. First, notice that linear association doesn't depend on the units of measurement – we might as well measure both variables in standard units. 

In [6]:
def standard_units(xyz):
    "Convert any array of numbers to standard units."
    # POLARS_CHANGE: .mean() and .std(ddof=0) work on a NumPy array and on a Polars Series (np.std is ddof=0).
    return (xyz - xyz.mean())/xyz.std(ddof=0)  

In [7]:
heights_SU = pl.DataFrame([
    standard_units(heights.get_column('MidParent')).alias('MidParent SU'),
    standard_units(heights.get_column('Child')).alias('Child SU')
])
heights_SU

MidParent SU,Child SU
f64,f64
3.454652,1.804156
3.454652,0.686005
3.454652,0.630097
3.454652,0.630097
2.472085,1.888017
…,…
-1.424873,-0.767591
-1.424873,-1.326667
-1.424873,-1.606205


On this scale, we can calculate our predictions exactly as before. But first we have to figure out how to convert our old definition of "close" points to a value on the new scale. We had said that midparent heights were "close" if they were within 0.5 inches of each other. Since standard units measure distances in units of SDs, we have to figure out how many SDs of midparent height correspond to 0.5 inches.

One SD of midparent heights is about 1.8 inches. So 0.5 inches is about 0.28 SDs.

In [8]:
sd_midparent = heights.to_series(0).std(ddof=0)
sd_midparent

1.8014050969207605

In [9]:
0.5/sd_midparent

0.2775611109653665

 We are now ready to modify our prediction function to make predictions on the standard units scale. All that has changed is that we are using the table of values in standard units, and defining "close" as above.

In [10]:
def predict_child_su(mpht_su):
    """Return a prediction of the height (in standard units) of a child 
    whose parents have a midparent height of mpht_su in standard units.
    """
    close = 0.5/sd_midparent
    close_points = heights_SU.filter(pl.col('MidParent SU').is_between(mpht_su-close, mpht_su + close, closed="left"))
    return close_points.get_column('Child SU').mean()   

In [11]:
heights_with_su_predictions = heights_SU.with_columns(
    heights_SU.get_column('MidParent SU').map_elements(predict_child_su, return_dtype=float).alias('Prediction SU')
    )

In [12]:
heights_with_su_predictions.hvplot.scatter(x='MidParent SU', y=['Child SU', 'Prediction SU'], color=['#002fa7', '#706700'])

Scatterplot with 'MidParent SU' on the x-axis. The shape of the data points is the same as before.In dark blue data points are labeled 'Child SU.' These dark blue data points tend to have a positive correlation; as the x value increases so does the y value. There are gold data points labeled 'Prediction SU.' The gold data points are in the center of the blue data points and also show a positive association. The x-axis ranges from -3 to 3. The y-axis ranges from -3 to 3.

This plot looks exactly like the plot drawn on the original scale. Only the numbers on the axes have changed. This confirms that we can understand the prediction process by just working in standard units.

## Identifying the Line in Standard Units
The scatter plot above has a *football* shape – that is, it is roughly oval like an American football. Not all scatter plots are football shaped, not even those that show linear association. But in this section we will work only with football shaped scatter plots. In the next section, we will generalize our analysis to other shapes of plots.

Here is a football shaped scatter plot with both variables measured in standard units. The 45 degree line is shown in red.

In [13]:
r = 0.5
x_demo = np.random.normal(0, 1, 10000)
z_demo = np.random.normal(0, 1, 10000)
y_demo = r*x_demo + np.sqrt(1 - r**2)*z_demo
(
    hv.Scatter((x_demo, y_demo)).opts(size=3, color='#002fa7')
    # * hv.Curve([(-4, -4*0.6), (4, 4*0.6)]).opts(color='green', line_width=2)
    * hv.Curve([(-4, -4), (4, 4)]).opts(color='red', line_width=2)
    # * hv.Curve([(1.5, -4), (1.5, 4)]).opts(color='black', line_width=2)
).opts(xlim=(-4, 4), ylim=(-4, 4), width=450, height=450,
        xlabel='x in standard units', ylabel='y in standard units')

Scatterplot with 'x in standard units' on the x-axis and 'y in standard units' on the y-axis. Both axes range from -4 to 4. The data points show a moderately strong positive correlation; as the x value increases so does the y value. A red line y=x is drawn and sits roughly in the middle of the data points.

But the 45 degree line is not the line that picks off the centers of the vertical strips. You can see that in the figure below, where the vertical line at 1.5 standard units is shown in black. The points on the scatter plot near the black line all have heights roughly in the -2 to 3 range. The red line is too high to pick off the center.

In [14]:
r = 0.5
x_demo = np.random.normal(0, 1, 10000)
z_demo = np.random.normal(0, 1, 10000)
y_demo = r*x_demo + np.sqrt(1 - r**2)*z_demo
(
    hv.Scatter((x_demo, y_demo)).opts(size=3, color='#002fa7')
    # * hv.Curve([(-4, -4*0.6), (4, 4*0.6)]).opts(color='green', line_width=2)
    * hv.Curve([(-4, -4), (4, 4)]).opts(color='red', line_width=2)
    * hv.Curve([(1.5, -4), (1.5, 4)]).opts(color='black', line_width=2)
).opts(xlim=(-4, 4), ylim=(-4, 4), width=450, height=450,
        xlabel='x in standard units', ylabel='y in standard units')

The previous graph is reproduced with the addition of a black vertical line y=1.5. The previous graph description: Scatterplot with 'x in standard units' on the x-axis and 'y in standard units' on the y-axis. Both axes range from -4 to 4. The data points show a moderately strong positive correlation; as the x value increases so does the y value. A red line y=x is drawn and sits roughly in the middle of the data points.

So the 45 degree line is not the "graph of averages." That line is the green one shown below.

In [15]:
r = 0.5
x_demo = np.random.normal(0, 1, 10000)
z_demo = np.random.normal(0, 1, 10000)
y_demo = r*x_demo + np.sqrt(1 - r**2)*z_demo
(
    hv.Scatter((x_demo, y_demo)).opts(size=3, color='#002fa7')
    * hv.Curve([(-4, -4*0.6), (4, 4*0.6)]).opts(color='green', line_width=2)
    * hv.Curve([(-4, -4), (4, 4)]).opts(color='red', line_width=2)
    * hv.Curve([(1.5, -4), (1.5, 4)]).opts(color='black', line_width=2)
).opts(xlim=(-4, 4), ylim=(-4, 4), width=450, height=450,
        xlabel='x in standard units', ylabel='y in standard units')

The previous graph is reproduced, including the black vertical line at y=1.5, with the addition of a green line that truly sits in the center of the data. Previous description: Scatterplot with 'x in standard units' on the x-axis and 'y in standard units' on the y-axis. Both axes range from -4 to 4. The data points show a moderately strong positive correlation; as the x value increases so does the y value. A red line y=x is drawn and sits roughly in the middle of the data points.

Both lines go through the origin (0, 0). The green line goes through the centers of the vertical strips (at least roughly), and is *flatter* than the red 45 degree line. 

The slope of the 45 degree line is 1. So the slope of the green "graph of averages" line is a value that is positive but less than 1. 

What value could that be? You've guessed it – it's $r$.

## The Regression Line, in Standard Units
The green "graph of averages" line is called the *regression line*, for a reason that we will explain shortly. But first, let's simulate some football shaped scatter plots with different values of $r$, and see how the line changes. In each case, the red 45 degree line has been drawn for comparison.

The function that performs the simulation is called `regression_line` and takes $r$ as its argument.

In [16]:

def regression_line(r):
    x = np.random.normal(0, 1, 10000)
    z = np.random.normal(0, 1, 10000)
    y = r*x + (np.sqrt(1-r**2))*z
    if r >= 0:
        diagonal = hv.Curve([(-4, -4), (4, 4)])
    else:
        diagonal = hv.Curve([(-4, 4), (4, -4)])
    return (
        hv.Scatter((x, y)).opts(color='#002fa7')
        * hv.Curve([(-4, -4*r), (4, 4*r)]).opts(color='green', line_width=2)
        * diagonal.opts(color='red', line_width=2)
    ).opts(xlim=(-4, 4), ylim=(-4, 4), width=450, height=450)

In [17]:
regression_line(0.95)

Scatterplot where both axes extend from -4 to 4. The data is highly strongly positively correlated; as the x values increase so do the y values. Two lines are drawn. A red line starts a little lower than the green line and ends a little higher than the green line. Both lines stay within the shape of the data. The red line is y=x.

In [18]:
regression_line(0.6)

Scatterplot where both axes extend from -4 to 4. The data points are moderately potively correlated; as the x values increase do the y avlues. The line y=x is drawn in red. There is also a green line with a smaller slope that better aligns with the data points.

When $r$ is close to 1, the scatter plot, the 45 degree line, and the regression line are all very close to each other. But for more moderate values of $r$, the regression line is noticeably flatter.

## The Regression Effect
In terms of prediction, this means that for a parents whose midparent height is at 1.5 standard units, our prediction of the child's height is somewhat *less* than 1.5 standard units. If the midparent height is 2 standard units, we predict that the child's height will be somewhat less than 2 standard units.

In other words, we predict that the child will be somewhat closer to average than the parents were. This is called "regression to the mean" and it is how the name *regression* arises.

Regression to the mean also works when the midparent height is below average. Children whose midparent heights were below average turned out to be somewhat taller relative to their generation, on average. 

In general, individuals who are away from average on one variable are expected to be not quite as far away from average on the other. This is called the *regression effect*.

Keep in mind that the regression effect is a statement about averages. For example, it says that if you take all children whose midparent height is 1.5 standard units, then the average height of these children is somewhat less than 1.5 standard units. It doesn't say that all of these children will be somewhat less than 1.5 standard units in height. Some will be taller, and some will be shorter. The *average* of these heights will be less than 1.5 standard units.

## The Equation of the Regression Line
In regression, we use the value of one variable (which we will call $x$) to predict the value of another (which we will call $y$). When the variables $x$ and $y$ are measured in standard units, the regression line for predicting $y$ based on $x$ has slope $r$ and passes through the origin. Thus the equation of the regression line can be written as:

$$
\text{estimate of }y ~=~ r \cdot x ~~~
\text{when both variables are measured in standard units}
$$

In the original units of the data, this becomes

$$
\frac{\text{estimate of}~y ~-~\text{average of}~y}{\text{SD of}~y}
~=~ r \times 
\frac{\text{the given}~x ~-~\text{average of}~x}{\text{SD of}~x}
$$


![Two figures are shown. The first is labeled "Regression Line in Standard Units." We see the point (0,0) with a horizontal line extending to the right. The length of the horizontal line is 1. From the end of that horizontal line is a veritcal line that extends upward with length r. The regression line is a sloping line that connects (and extends beyond in both directions) from (0,0) to the top of the vertical line. The second figure is labeled "Regression Line in Original Units." A similar figure is shown with different annotations. (0,0) is now (average of x, average of y), the horizontal length is now "SD of x" and the vertical length is now "r SD of y"](../../../images/regline.png)

The slope and intercept of the regression line in original units can be derived from the diagram above. 

$$
\mathbf{\text{slope of the regression line}} ~=~ r \cdot
\frac{\text{SD of }y}{\text{SD of }x}
$$

$$
\mathbf{\text{intercept of the regression line}} ~=~
\text{average of }y ~-~ \text{slope} \cdot \text{average of }x
$$

The three functions below compute the correlation, slope, and intercept. All of them take three arguments: the name of the table, the label of the column containing $x$, and the label of the column containing $y$.

In [19]:
def correlation(t, label_x, label_y):
    return (standard_units(t.get_column(label_x))*standard_units(t.get_column(label_y))).mean()

def slope(t, label_x, label_y):
    r = correlation(t, label_x, label_y)
    return r*t.get_column(label_y).std(ddof=0)/t.get_column(label_x).std(ddof=0)

def intercept(t, label_x, label_y):
    return t.get_column(label_y).mean() - slope(t, label_x, label_y)*t.get_column(label_x).mean()

## The Regression Line in the Units of the Data
The correlation between midparent height and child's height is 0.32:

In [20]:
family_r = correlation(heights, 'MidParent', 'Child')
family_r

0.32094989606395874

We can also find the equation of the regression line for predicting the child's height based on midparent height.

In [21]:
family_slope = slope(heights, 'MidParent', 'Child')
family_intercept = intercept(heights, 'MidParent', 'Child')
family_slope, family_intercept

(0.6373608969694768, 22.636240549589907)

The equation of the regression line is

$$
\text{estimate of child's height} ~=~ 0.64 \cdot \text{midparent height} ~+~ 22.64
$$

This is also known as the *regression equation.* The principal use of the regression equation is to predict $y$ based on $x$.

For example, for a midparent height of 70.48 inches, the regression equation predicts the child's height to be 67.56 inches.

In [22]:
family_slope * 70.48 + family_intercept

67.55743656799864

Our original prediction, created by taking the average height of all children who had midparent heights close to 70.48, came out to be pretty close: 67.63 inches compared to the regression line's prediction of 67.55 inches.

In [23]:
heights_with_predictions.filter(pl.col('MidParent') == 70.48).show(3)

MidParent,Child,Prediction
f64,f64,f64
70.48,74.0,67.634239
70.48,70.0,67.634239
70.48,68.0,67.634239


Here are all of the rows of the table, along with our original predictions and the new regression predictions of the children's heights.

In [24]:
heights_with_predictions = heights_with_predictions.with_columns(
    (family_slope * heights.get_column('MidParent') + family_intercept).alias('Regression Prediction')
)
heights_with_predictions

MidParent,Child,Prediction,Regression Prediction
f64,f64,f64,f64
75.43,73.2,70.1,70.712373
75.43,69.2,70.1,70.712373
75.43,69.0,70.1,70.712373
75.43,69.0,70.1,70.712373
73.66,73.5,70.415789,69.584244
…,…,…,…
66.64,64.0,65.156579,65.109971
66.64,62.0,65.156579,65.109971
66.64,61.0,65.156579,65.109971


In [25]:
heights_with_predictions.hvplot.scatter(x='MidParent', y=['Child', 'Prediction', 'Regression Prediction'], color=['#002fa7', '#706700', '#006a94'])

Scatterplot with 'MidParent' on the x-axis. Three colors of data points are present. In dark blue is 'Child.' These data points have a moderate positive correlation and are clearly blob-like, with multiple points possible per x-value. The gold data points are labeled 'Prediction' and the light blue data points are labeled 'Regression Prediction.' Each of the gold and light blue data points have one point per possible x value. The light blue and gold data points are very close together, but we see the regression prediction data points falling along a straight line, where the gold data points have some wavy-ness to them.

The light blue dots show the regression predictions, all on the regression line. Notice how the line is very close to the gold graph of averages. For these data, the regression line does a good job of approximating the centers of the vertical strips.

## Fitted Values
The predictions all lie on the line and are known as the "fitted values". The function `fit` takes the name of the table and the labels of $x$ and $y$, and returns a Series of fitted values, one fitted value for each point in the scatter plot.

In [26]:
def fit(table, x, y):
    """Return the height of the regression line at each x value."""
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * table.get_column(x) + b

It is easier to see the line in the graph below than in the one above.

In [27]:
heights.with_columns(fit(heights, 'MidParent', 'Child').alias('Fitted')).hvplot.scatter(x='MidParent', y=['Child', 'Fitted'], color=['#002fa7', '#706700'])

The previous scatterplot is reproduced without the light blue data points. The 'Fitted' data is now in gold and 'Child' data remains in dark blue.

Another way to draw the line is to overlay it on the scatter plot with HoloViews `hv.Slope`, which takes the slope and the intercept. (POLARS_CHANGE: Data 8 used the option `fit_line=True` with the Table method `scatter`; hvPlot has no `fit_line` option.)

In [28]:
# POLARS_CHANGE: hvplot has no fit_line option; hv.Slope draws the line with the given slope and intercept.
(
    heights.hvplot.scatter(x='MidParent', y='Child')
    * hv.Slope(slope(heights, 'MidParent', 'Child'), intercept(heights, 'MidParent', 'Child')).opts(color='#002fa7', line_width=2)
)

The previous scatterplot is reproduced with the gold data points replaced by a dark blue line. The line cuts through the center of the data points.

## Units of Measurement of the Slope
The slope is a ratio, and it worth taking a moment to study the units in which it is measured. Our example comes from the familiar dataset about mothers who gave birth in a hospital system. The scatter plot of pregnancy weights versus heights looks like a football that has been used in one game too many, but it's close enough to a football that we can justify putting our fitted line through it. In later sections we will see how to make such justifications more formal.

In [29]:
baby = pl.read_csv(path_data + 'baby.csv')

In [30]:
(
    baby.hvplot.scatter(x='Maternal Height', y='Maternal Pregnancy Weight')
    * hv.Slope(
        slope(baby, 'Maternal Height', 'Maternal Pregnancy Weight'),
        intercept(baby, 'Maternal Height', 'Maternal Pregnancy Weight'),
    ).opts(color='#002fa7', line_width=2)
)

Scatterplot with 'Maternal Height' on the x-axis and 'Maternal Pregnancy Weight' on the y-axis. Data points have discrete x values, and we see many possible y values for the majority of each x value. But generally, as x values increase so do y values. There are some notable outliers with higher y values than the other data points with the same x value. A blue line with a positive slope is drawn through the data.

In [31]:
slope(baby, 'Maternal Height', 'Maternal Pregnancy Weight')

3.572846259275056

The slope of the regression line is **3.57 pounds per inch**. This means that for two women who are 1 inch apart in height, our prediction of pregnancy weight will differ by 3.57 pounds. For a woman who is 2 inches taller than another, our prediction of pregnancy weight will be

$$
2 \times 3.57 ~=~ 7.14
$$

pounds more than our prediction for the shorter woman.

Notice that the successive vertical strips in the scatter plot are one inch apart, because the heights have been rounded to the nearest inch. Another way to think about the slope is to take any two consecutive strips (which are necessarily 1 inch apart), corresponding to two groups of women who are separated by 1 inch in height. The slope of 3.57 pounds per inch means that the average pregnancy weight of the taller group is about 3.57 pounds more than that of the shorter group.


## Example

Suppose that our goal is to use regression to estimate the height of a basset hound based on its weight, using a sample that looks consistent with the regression model. Suppose the observed correlation $r$ is 0.5, and that the summary statistics for the two variables are as in the table below: 

|       |**average**    |**SD**     |   
|------:|:-------------:|:---------:|
|height |14 inches      |2 inches   |
|weight |50 pounds      |5 pounds   |

To calculate the equation of the regression line, we need the slope and the intercept.

$$
\text{slope} ~=~ \frac{r \cdot \text{SD of }y}{\text{SD of }x} ~=~
\frac{0.5 \cdot 2 \text{ inches}}{5 \text{ pounds}} ~=~ 0.2 ~\text{inches per pound}
$$


$$
\text{intercept} ~=~ \text{average of }y - \text{slope}\cdot \text{average of } x
~=~ 14 \text{ inches} ~-~ 0.2 \text{ inches per pound} \cdot 50 \text{ pounds}
~=~ 4 \text{ inches}
$$

The equation of the regression line allows us to calculate the estimated height, in inches,
based on a given weight in pounds:

$$
\text{estimated height} ~=~ 0.2 \cdot \text{given weight} ~+~ 4
$$

The slope of the line is measures the increase in the estimated height per unit increase in weight. The slope is positive, and it is important to note that this does not mean that we think basset hounds get taller if they put on weight. The slope reflects the difference in the average heights of two groups of dogs that are 1 pound apart in weight. Specifically, consider a group of dogs whose weight is $w$ pounds, and the group whose weight is $w+1$ pounds. The second group is estimated to be 0.2 inches taller, on average. This is true for all values of $w$ in the sample.

In general, the slope of the regression line can be interpreted as the average increase in $y$ per unit increase in $x$. Note that if the slope is negative, then for every unit increase in $x$, the average of $y$ decreases.

## Endnote
Even though we won't establish the mathematical basis for the regression equation, we can see that it gives pretty good predictions when the scatter plot is football shaped. It is a surprising mathematical fact that no matter what the shape of the scatter plot, the same equation gives the "best" among all straight lines. That's the topic of the next section.